# 🎵 Musical Note Analyser

A clean Google Colab version of the project.

This notebook:
- Loads a WAV audio file
- Converts stereo audio to mono
- Normalizes the signal
- Applies **Fourier Transform (FFT)** to each audio frame
- Uses **Harmonic Product Spectrum (HPS)** for pitch detection
- Maps detected frequencies to musical notes
- Estimates note start/end times
- Displays a spectrogram and frequency spectrum


In [ ]:
# Import libraries

import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
from google.colab import files


In [ ]:
# Upload your WAV audio file

uploaded = files.upload()
AUDIO_FILE = next(iter(uploaded))

print("Loaded:", AUDIO_FILE)


In [ ]:
# Musical note frequencies (A4 = 440 Hz)

NOTES = {
    "C": 261.63,
    "C#": 277.18,
    "D": 293.66,
    "D#": 311.13,
    "E": 329.63,
    "F": 349.23,
    "F#": 369.99,
    "G": 392.00,
    "G#": 415.30,
    "A": 440.00,
    "A#": 466.16,
    "B": 493.88
}


def closest_note(frequency):
    return min(NOTES, key=lambda note: abs(NOTES[note] - frequency))


In [ ]:
# Load and prepare the audio

def load_audio(audio_file):
    sample_rate, data = wav.read(audio_file)

    # Convert stereo to mono
    if data.ndim > 1:
        data = data.mean(axis=1)

    # Convert to float
    data = data.astype(float)

    # Normalize amplitude
    peak = np.max(np.abs(data))
    if peak > 0:
        data /= peak

    return sample_rate, data


sample_rate, data = load_audio(AUDIO_FILE)

print("Sample rate:", sample_rate, "Hz")
print("Duration:", round(len(data) / sample_rate, 2), "seconds")


In [ ]:
# Detect musical notes using FFT + HPS

def detect_notes(audio_file, frame_duration=0.06, silence_threshold=0.04):
    sample_rate, data = load_audio(audio_file)

    frame_size = int(frame_duration * sample_rate)
    window = np.hanning(frame_size)

    detected_notes = []

    for start in range(0, len(data) - frame_size + 1, frame_size):

        frame = data[start:start + frame_size]

        # Ignore silent frames
        if np.max(np.abs(frame)) < silence_threshold:
            continue

        # ---------------------------------------------------------
        # FOURIER TRANSFORM
        # Convert the audio from time domain to frequency domain
        # ---------------------------------------------------------
        spectrum = np.abs(np.fft.rfft(frame * window))
        frequencies = np.fft.rfftfreq(
            frame_size,
            1 / sample_rate
        )

        # HPS uses the frequency spectrum to find the fundamental
        hps = spectrum.copy()

        for harmonic in range(2, 4):
            downsampled = spectrum[::harmonic]
            hps[:len(downsampled)] *= downsampled

        # Only consider a useful musical frequency range
        valid = (frequencies >= 150) & (frequencies <= 900)
        hps[~valid] = 0

        peak = np.argmax(hps)
        frequency = frequencies[peak]

        if frequency <= 0:
            continue

        note = closest_note(frequency)

        start_time = start / sample_rate
        end_time = (start + frame_size) / sample_rate

        # Merge consecutive frames containing the same note
        if detected_notes and detected_notes[-1][0] == note:
            detected_notes[-1] = (
                note,
                frequency,
                detected_notes[-1][2],
                end_time
            )
        else:
            detected_notes.append(
                (note, frequency, start_time, end_time)
            )

    return detected_notes


In [ ]:
# Run note detection

detected_notes = detect_notes(AUDIO_FILE)

print("\nDetected Notes")
print("-" * 65)
print(f"{'Note':<8}{'Frequency':<15}{'Start (s)':<15}{'End (s)':<15}")
print("-" * 65)

for note, frequency, start, end in detected_notes:
    print(f"{note:<8}{frequency:<15.2f}{start:<15.2f}{end:<15.2f}")


In [ ]:
# Plot spectrogram

plt.figure(figsize=(12, 5))

plt.specgram(
    data,
    Fs=sample_rate,
    NFFT=2048,
    noverlap=1024
)

plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.title("Audio Spectrogram")
plt.colorbar(label="Intensity (dB)")
plt.tight_layout()
plt.show()


In [ ]:
# Plot frequency spectrum

spectrum = np.abs(np.fft.rfft(data))
frequencies = np.fft.rfftfreq(
    len(data),
    1 / sample_rate
)

plt.figure(figsize=(12, 5))
plt.plot(frequencies, spectrum)

plt.xlim(0, 1000)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.title("Frequency Spectrum")
plt.tight_layout()
plt.show()
